# 01 — Data Understanding & Profiling

**Project:** Purchase-to-Pay Process Mining & Exception Intelligence

## Phase 2 objective

Before cleaning, modeling, SQL analysis, or dashboard development, this notebook validates the source event log and answers five questions:

1. What is the physical structure of the dataset?
2. What is the analytical grain?
3. Which fields are complete, sparse, constant, or high-cardinality?
4. Which activities and case patterns actually exist?
5. Which source fields should later be **kept, transformed, dropped, or investigated**?

> **Rule:** Raw data is profiled first. No rows are deleted and no business rule is imposed in this notebook without evidence.


## 1. Environment setup

Only standard analysis libraries are used at this stage.  
We intentionally avoid process-mining or machine-learning libraries until the source data has been validated.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("/mnt/data/output.csv")
assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"

print(f"Source file: {DATA_PATH.name}")
print(f"File size: {DATA_PATH.stat().st_size / 1024**2:,.1f} MB")


Source file: output.csv
File size: 423.9 MB


## 2. Load the raw event log

At this stage the CSV is loaded **without dropping duplicates, filling nulls, or renaming source columns**.

That preserves the original evidence for profiling.


In [2]:
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Rows   : {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Memory : {df.memory_usage(deep=True).sum() / 1024**3:,.2f} GB")


Rows   : 1,595,923
Columns: 21


Memory : 1.54 GB


## 3. Structural inspection

A Purchase-to-Pay event log is not a normal one-row-per-transaction table.

- `case:concept:name` identifies the **process case**
- `concept:name` identifies the **event/activity**
- `time:timestamp` tells us when the event occurred

Therefore, the same case ID is expected to appear on multiple rows.


In [3]:
display(df.head())

print("\nColumn names:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")


,User,org:resource,concept:name,Cumulative net worth (EUR),time:timestamp,case:Spend area text,case:Company,case:Document Type,case:Sub spend area text,case:Purchasing Document,case:Purch. Doc. Category name,case:Vendor,case:Item Type,case:Item Category,case:Spend classification text,case:Source,case:Name,case:GR-Based Inv. Verif.,case:Item,case:concept:name,case:Goods Receipt
0,batch_00,batch_00,SRM: Created,298.00,2018-01-02 12:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,Purchase order,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,1,2000000000_00001,True
1,batch_00,batch_00,SRM: Complete,298.00,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,Purchase order,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,1,2000000000_00001,True
2,batch_00,batch_00,SRM: Awaiting Approval,298.00,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,Purchase order,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,1,2000000000_00001,True
3,batch_00,batch_00,SRM: Document Completed,298.00,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,Purchase order,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,1,2000000000_00001,True
4,batch_00,batch_00,SRM: In Transfer to Execution Syst.,298.00,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,Purchase order,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,1,2000000000_00001,True



Column names:
01. User
02. org:resource
03. concept:name
04. Cumulative net worth (EUR)
05. time:timestamp
06. case:Spend area text
07. case:Company
08. case:Document Type
09. case:Sub spend area text
10. case:Purchasing Document
11. case:Purch. Doc. Category name
12. case:Vendor
13. case:Item Type
14. case:Item Category
15. case:Spend classification text
16. case:Source
17. case:Name
18. case:GR-Based Inv. Verif.
19. case:Item
20. case:concept:name
21. case:Goods Receipt


In [4]:
baseline = pd.Series({
    "Event rows": len(df),
    "Columns": df.shape[1],
    "Unique P2P cases": df["case:concept:name"].nunique(dropna=True),
    "Purchasing documents": df["case:Purchasing Document"].nunique(dropna=True),
    "Vendors": df["case:Vendor"].nunique(dropna=True),
    "Activities": df["concept:name"].nunique(dropna=True),
    "Companies": df["case:Company"].nunique(dropna=True),
}, name="Value").to_frame()

display(baseline)


,Value
Event rows,1595923
Columns,21
Unique P2P cases,251734
Purchasing documents,76349
Vendors,1975
Activities,42
Companies,4


### Interpretation

The number of event rows is much larger than the number of cases because one P2P case contains multiple business events.

This distinction becomes critical later when calculating:

- exception rates,
- vendor rates,
- cycle-time KPIs,
- case counts,
- and financial measures.

Using event rows as the denominator for case-level KPIs would produce incorrect results.


## 4. Column-level profiling

For every source field we calculate:

- pandas data type,
- non-null count,
- missing count and percentage,
- distinct-value count,
- distinct percentage,
- and a few observed values.

This provides the evidence for later **KEEP / TRANSFORM / DROP / INVESTIGATE** decisions.


In [5]:
def profile_columns(data):
    records = []
    n = len(data)

    for col in data.columns:
        non_null = data[col].notna().sum()
        missing = n - non_null
        nunique = data[col].nunique(dropna=True)

        sample_values = (
            data[col]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(3)
            .tolist()
        )

        records.append({
            "column": col,
            "dtype": str(data[col].dtype),
            "non_null": non_null,
            "missing": missing,
            "missing_pct": round(missing / n * 100, 2),
            "nunique": nunique,
            "unique_pct": round(nunique / n * 100, 4),
            "sample_values": " | ".join(sample_values)
        })

    return pd.DataFrame(records)

column_profile = profile_columns(df)
display(column_profile)


,column,dtype,non_null,missing,missing_pct,nunique,unique_pct,sample_values
0,User,object,1595923,0,0.00,628,0.04,batch_00 | user_000 | NONE
1,org:resource,object,1595923,0,0.00,628,0.04,batch_00 | user_000 | NONE
2,concept:name,object,1595923,0,0.00,42,0.00,SRM: Created | SRM: Complete | SRM: Awaiting A...
3,Cumulative net worth (EUR),float64,1595923,0,0.00,25221,1.58,298.0 | 557.0 | 76210.0
4,time:timestamp,object,1595923,0,0.00,167432,10.49,2018-01-02 12:53:00+00:00 | 2018-01-02 13:53:0...
5,case:Spend area text,object,1579629,16294,1.02,20,0.00,CAPEX & SOCS | Marketing | Enterprise Services
6,case:Company,object,1595923,0,0.00,4,0.00,companyID_0000 | companyID_0001 | companyID_0002
7,case:Document Type,object,1595923,0,0.00,3,0.00,EC Purchase order | Standard PO | Framework order
8,case:Sub spend area text,object,1579629,16294,1.02,135,0.01,Facility Management | Marketing Support Servic...
9,case:Purchasing Document,int64,1595923,0,0.00,76349,4.78,2000000000 | 2000000001 | 2000000002


## 5. Missing-value profile

Missingness is not automatically treated as bad data.

In event logs, some attributes may only apply to particular process types or events.  
The next step is therefore to identify **where** missingness exists before deciding **why** it exists.


In [6]:
missing_profile = (
    column_profile[["column", "missing", "missing_pct"]]
    .sort_values(["missing_pct", "missing"], ascending=False)
    .reset_index(drop=True)
)

display(missing_profile)


,column,missing,missing_pct
0,case:Spend area text,16294,1.02
1,case:Sub spend area text,16294,1.02
2,case:Spend classification text,16294,1.02
3,User,0,0.00
4,org:resource,0,0.00
5,concept:name,0,0.00
6,Cumulative net worth (EUR),0,0.00
7,time:timestamp,0,0.00
8,case:Company,0,0.00
9,case:Document Type,0,0.00


## 6. Validate event-level vs case-level grain

We now inspect how many events belong to each case.

This helps establish the distribution of case complexity before any exception rule is created.


In [7]:
events_per_case = df.groupby("case:concept:name", dropna=False).size()

case_event_summary = events_per_case.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame("events_per_case")

display(case_event_summary)

print(f"Cases with exactly 1 event: {(events_per_case == 1).sum():,}")
print(f"Maximum events in one case: {events_per_case.max():,}")


,events_per_case
count,"251,734.00"
mean,6.34
std,13.06
min,1.00
25%,5.00
50%,5.00
75%,6.00
90%,7.00
95%,9.00
99%,24.00


Cases with exactly 1 event: 2,835
Maximum events in one case: 990


## 7. Activity inventory

Event frequency alone is not enough.

For each activity we also count the number of **unique cases containing that activity**.  
This prevents a repeatedly occurring event in a small number of cases from being mistaken for a common case-level behavior.


In [8]:
activity_profile = (
    df.groupby("concept:name")
      .agg(
          event_count=("concept:name", "size"),
          case_count=("case:concept:name", "nunique")
      )
      .sort_values("event_count", ascending=False)
      .reset_index()
)

activity_profile["event_share_pct"] = (
    activity_profile["event_count"] / len(df) * 100
).round(2)

activity_profile["case_share_pct"] = (
    activity_profile["case_count"] / df["case:concept:name"].nunique() * 100
).round(2)

display(activity_profile)


,concept:name,event_count,case_count,event_share_pct,case_share_pct
0,Record Goods Receipt,314097,234479,19.68,93.15
1,Create Purchase Order Item,251734,251734,15.77,100.00
2,Record Invoice Receipt,228760,211379,14.33,83.97
3,Vendor creates invoice,219919,209946,13.78,83.40
4,Clear Invoice,194393,183677,12.18,72.96
5,Record Service Entry Sheet,164975,5604,10.34,2.23
6,Remove Payment Block,57136,55839,3.58,22.18
7,Create Purchase Requisition Item,46592,46592,2.92,18.51
8,Receive Order Confirmation,32065,32061,2.01,12.74
9,Change Quantity,21449,17590,1.34,6.99


## 8. Phase 2 profiling checkpoint

At this point we have established the basic dataset structure and analytical grain.

The next sections of this notebook will investigate:

1. case-level attribute consistency,
2. duplicate-looking rows,
3. timestamp anomalies,
4. `User` vs `org:resource`,
5. monetary-value behavior,
6. process-category distributions,
7. activity taxonomy,
8. and final source-column disposition.

No source record has been deleted or modified yet.
